# Honest Evaluation - V2 vs V3 Comparison

This notebook provides an **honest assessment** of our intent classifier:

## Evaluations:
1. **V2 Model** - Trained on data with leakage (same products in train/test)
2. **V3 Model** - Trained on proper data (OOD products in test)
3. **Manual Test Set** - 50 hand-crafted realistic queries

## Purpose:
- Demonstrate the impact of data leakage on reported accuracy
- Show realistic performance expectations
- Identify failure modes and improvement areas

In [ ]:
# Cell 1: Imports and Setup
import os
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path

import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import matplotlib.pyplot as plt
import seaborn as sns

# Project paths
PROJECT_ROOT = Path(os.getcwd()).parent
DATA_DIR = PROJECT_ROOT / "data" / "synthetic"
MODEL_V2_DIR = PROJECT_ROOT / "models" / "intent_classifier_v2" / "best_model"
MODEL_V3_DIR = PROJECT_ROOT / "models" / "intent_classifier_v3" / "best_model"
OUTPUT_DIR = PROJECT_ROOT / "models" / "evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"V2 Model: {MODEL_V2_DIR}")
print(f"V3 Model: {MODEL_V3_DIR}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# Cell 2: Load Manual Test Set

print("Loading manual test set...")
print("=" * 60)

manual_test_path = DATA_DIR / "manual_test_50.csv"

if not manual_test_path.exists():
    print(f"ERROR: Manual test set not found at {manual_test_path}")
    print("Please ensure manual_test_50.csv exists.")
else:
    manual_df = pd.read_csv(manual_test_path)
    print(f"Loaded {len(manual_df)} manual test samples")
    print(f"\nDifficulty distribution:")
    print(manual_df['difficulty'].value_counts())
    print(f"\nIntent distribution:")
    print(manual_df['intent'].value_counts())

In [ ]:
# Cell 3: Load Models

print("Loading models...")
print("=" * 60)

models = {}

# Load V2 if exists
if MODEL_V2_DIR.exists():
    print("Loading V2 model...")
    models['v2'] = pipeline(
        "text-classification",
        model=str(MODEL_V2_DIR),
        device=-1
    )
    print("  V2 loaded!")
else:
    print("  V2 model not found - skipping")

# Load V3 if exists
if MODEL_V3_DIR.exists():
    print("Loading V3 model...")
    models['v3'] = pipeline(
        "text-classification",
        model=str(MODEL_V3_DIR),
        device=-1
    )
    print("  V3 loaded!")
else:
    print("  V3 model not found - skipping")

print(f"\nModels available: {list(models.keys())}")

In [ ]:
# Cell 4: Define Evaluation Function

def evaluate_model(model, test_df, model_name):
    """Evaluate a model on test set and return results"""
    predictions = []
    confidences = []
    
    for text in test_df['text']:
        result = model(text)[0]
        predictions.append(result['label'])
        confidences.append(result['score'])
    
    # Calculate metrics
    true_labels = test_df['intent'].tolist()
    accuracy = accuracy_score(true_labels, predictions)
    
    correct = [p == t for p, t in zip(predictions, true_labels)]
    
    return {
        'model_name': model_name,
        'predictions': predictions,
        'confidences': confidences,
        'correct': correct,
        'accuracy': accuracy,
        'true_labels': true_labels
    }

print("Evaluation function defined!")

In [ ]:
# Cell 5: Evaluate on Manual Test Set

print("\n" + "=" * 60)
print("MANUAL TEST SET EVALUATION")
print("=" * 60)
print("Testing on 50 hand-crafted realistic queries...")
print()

results = {}

for model_name, model in models.items():
    print(f"\nEvaluating {model_name.upper()} model...")
    results[model_name] = evaluate_model(model, manual_df, model_name)
    print(f"  Accuracy: {results[model_name]['accuracy']:.2%}")

# Compare
if len(results) > 1:
    print("\n" + "-" * 40)
    print("COMPARISON:")
    for name, res in results.items():
        print(f"  {name.upper()}: {res['accuracy']:.2%}")

In [ ]:
# Cell 6: Detailed Results by Difficulty

print("\n" + "=" * 60)
print("RESULTS BY DIFFICULTY LEVEL")
print("=" * 60)

for model_name, res in results.items():
    print(f"\n{model_name.upper()} Model:")
    print("-" * 40)
    
    for difficulty in ['easy', 'medium', 'hard']:
        mask = manual_df['difficulty'] == difficulty
        if mask.sum() > 0:
            diff_correct = [c for c, m in zip(res['correct'], mask) if m]
            diff_acc = sum(diff_correct) / len(diff_correct)
            print(f"  {difficulty.capitalize()}: {diff_acc:.2%} ({sum(diff_correct)}/{len(diff_correct)})")

In [ ]:
# Cell 7: Detailed Results by Intent

print("\n" + "=" * 60)
print("RESULTS BY INTENT")
print("=" * 60)

intents = manual_df['intent'].unique()

for model_name, res in results.items():
    print(f"\n{model_name.upper()} Model:")
    print("-" * 40)
    
    for intent in intents:
        mask = manual_df['intent'] == intent
        if mask.sum() > 0:
            intent_correct = [c for c, m in zip(res['correct'], mask) if m]
            intent_acc = sum(intent_correct) / len(intent_correct)
            print(f"  {intent}: {intent_acc:.2%} ({sum(intent_correct)}/{len(intent_correct)})")

In [ ]:
# Cell 8: Classification Reports

print("\n" + "=" * 60)
print("CLASSIFICATION REPORTS (Manual Test Set)")
print("=" * 60)

for model_name, res in results.items():
    print(f"\n{model_name.upper()} Model:")
    print("-" * 60)
    print(classification_report(res['true_labels'], res['predictions'], digits=3))

In [ ]:
# Cell 9: Error Analysis

print("\n" + "=" * 60)
print("ERROR ANALYSIS")
print("=" * 60)

for model_name, res in results.items():
    print(f"\n{model_name.upper()} Model Errors:")
    print("-" * 60)
    
    error_count = 0
    for i, (text, true_label, pred, conf, correct) in enumerate(
        zip(manual_df['text'], res['true_labels'], res['predictions'], res['confidences'], res['correct'])
    ):
        if not correct:
            error_count += 1
            difficulty = manual_df.iloc[i]['difficulty']
            notes = manual_df.iloc[i]['notes']
            print(f"\n{error_count}. \"{text}\"")
            print(f"   True: {true_label} | Pred: {pred} (conf: {conf:.3f})")
            print(f"   Difficulty: {difficulty} | Notes: {notes}")
    
    print(f"\nTotal errors: {error_count}/{len(manual_df)}")

In [ ]:
# Cell 10: Confidence Calibration

print("\n" + "=" * 60)
print("CONFIDENCE CALIBRATION")
print("=" * 60)

fig, axes = plt.subplots(1, len(results), figsize=(7*len(results), 6))
if len(results) == 1:
    axes = [axes]

for ax, (model_name, res) in zip(axes, results.items()):
    correct_conf = [c for c, correct in zip(res['confidences'], res['correct']) if correct]
    wrong_conf = [c for c, correct in zip(res['confidences'], res['correct']) if not correct]
    
    ax.hist(correct_conf, bins=15, alpha=0.7, label=f'Correct (n={len(correct_conf)})', color='green')
    if wrong_conf:
        ax.hist(wrong_conf, bins=15, alpha=0.7, label=f'Wrong (n={len(wrong_conf)})', color='red')
    
    ax.set_xlabel('Confidence Score')
    ax.set_ylabel('Count')
    ax.set_title(f'{model_name.upper()} Confidence Distribution')
    ax.legend()
    
    # Print stats
    print(f"\n{model_name.upper()} Model:")
    print(f"  Correct mean conf: {np.mean(correct_conf):.4f}")
    if wrong_conf:
        print(f"  Wrong mean conf: {np.mean(wrong_conf):.4f}")
        print(f"  Wrong max conf: {np.max(wrong_conf):.4f} (overconfident!)")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'manual_test_confidence.png', dpi=300)
plt.show()

In [ ]:
# Cell 11: Confusion Matrix Comparison

intents_order = ['order_status', 'payment_info', 'product_price', 'product_stock', 'product_description', 'out_of_scope']

fig, axes = plt.subplots(1, len(results), figsize=(8*len(results), 7))
if len(results) == 1:
    axes = [axes]

for ax, (model_name, res) in zip(axes, results.items()):
    cm = confusion_matrix(res['true_labels'], res['predictions'], labels=intents_order)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=intents_order, yticklabels=intents_order)
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    ax.set_title(f'{model_name.upper()} - Manual Test Confusion Matrix\nAccuracy: {res["accuracy"]:.2%}')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'manual_test_confusion_matrix.png', dpi=300)
plt.show()

In [ ]:
# Cell 12: Load and Compare Previous Test Results

print("\n" + "=" * 60)
print("COMPARISON WITH PREVIOUS EVALUATIONS")
print("=" * 60)

comparison_data = []

# V2 training results
v2_summary_path = PROJECT_ROOT / "models" / "intent_classifier_v2" / "training_summary.json"
if v2_summary_path.exists():
    with open(v2_summary_path, 'r') as f:
        v2_summary = json.load(f)
    comparison_data.append({
        'Model': 'V2',
        'Test Set': 'V2 Test (same products)',
        'Accuracy': v2_summary.get('test_accuracy', 'N/A'),
        'Note': 'Data leakage - overfitted'
    })

# V3 training results
v3_summary_path = PROJECT_ROOT / "models" / "intent_classifier_v3" / "training_summary.json"
if v3_summary_path.exists():
    with open(v3_summary_path, 'r') as f:
        v3_summary = json.load(f)
    comparison_data.append({
        'Model': 'V3',
        'Test Set': 'V3 OOD Test (unseen products)',
        'Accuracy': v3_summary.get('ood_test_accuracy', 'N/A'),
        'Note': 'Honest evaluation'
    })

# Manual test results
for model_name, res in results.items():
    comparison_data.append({
        'Model': model_name.upper(),
        'Test Set': 'Manual (50 real queries)',
        'Accuracy': res['accuracy'],
        'Note': 'Real-world simulation'
    })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

In [ ]:
# Cell 13: Summary Visualization

print("\n" + "=" * 60)
print("ACCURACY COMPARISON CHART")
print("=" * 60)

if len(comparison_data) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Prepare data
    labels = [f"{d['Model']} on\n{d['Test Set'][:20]}..." if len(d['Test Set']) > 20 else f"{d['Model']} on\n{d['Test Set']}" for d in comparison_data]
    values = [float(d['Accuracy']) if isinstance(d['Accuracy'], (int, float)) else 0 for d in comparison_data]
    colors = ['red' if 'leakage' in d['Note'].lower() else 'green' for d in comparison_data]
    
    bars = ax.bar(range(len(labels)), values, color=colors, alpha=0.7)
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.1%}',
                ha='center', va='bottom', fontsize=12, fontweight='bold')
    
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title('Intent Classifier Accuracy Comparison\n(Red = Overfitted, Green = Honest)', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.axhline(y=0.85, color='gray', linestyle='--', alpha=0.5, label='Target: 85%')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'accuracy_comparison.png', dpi=300)
    plt.show()

In [ ]:
# Cell 14: Save Evaluation Report

print("\n" + "=" * 60)
print("SAVING EVALUATION REPORT")
print("=" * 60)

# Prepare detailed results
detailed_results = []
for model_name, res in results.items():
    for i, (text, true_label, pred, conf, correct) in enumerate(
        zip(manual_df['text'], res['true_labels'], res['predictions'], res['confidences'], res['correct'])
    ):
        detailed_results.append({
            'model': model_name,
            'text': text,
            'true_label': true_label,
            'predicted': pred,
            'confidence': conf,
            'correct': correct,
            'difficulty': manual_df.iloc[i]['difficulty'],
            'notes': manual_df.iloc[i]['notes']
        })

detailed_df = pd.DataFrame(detailed_results)
detailed_df.to_csv(OUTPUT_DIR / 'manual_test_detailed_results.csv', index=False)
print(f"Detailed results saved to: {OUTPUT_DIR / 'manual_test_detailed_results.csv'}")

# Save summary
summary = {
    'evaluation_date': pd.Timestamp.now().isoformat(),
    'manual_test_samples': len(manual_df),
    'models_evaluated': list(results.keys()),
    'results': {
        model_name: {
            'accuracy': res['accuracy'],
            'correct_count': sum(res['correct']),
            'error_count': len(res['correct']) - sum(res['correct']),
            'mean_confidence_correct': float(np.mean([c for c, correct in zip(res['confidences'], res['correct']) if correct])),
            'mean_confidence_wrong': float(np.mean([c for c, correct in zip(res['confidences'], res['correct']) if not correct])) if sum([not c for c in res['correct']]) > 0 else None
        }
        for model_name, res in results.items()
    },
    'comparison_table': comparison_data
}

with open(OUTPUT_DIR / 'evaluation_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f"Summary saved to: {OUTPUT_DIR / 'evaluation_summary.json'}")

In [ ]:
# Cell 15: Final Summary

print("\n" + "=" * 60)
print("HONEST EVALUATION - FINAL SUMMARY")
print("=" * 60)

print("""
KEY FINDINGS:
=============

1. DATA LEAKAGE IMPACT:
   - V2 reported ~98% accuracy, but this was inflated
   - Same products appeared in train and test sets
   - Model memorized patterns, not learned intent understanding

2. HONEST PERFORMANCE:
   - V3 with proper OOD evaluation shows realistic performance
   - Manual test set simulates real customer queries
   - Expect 80-90% accuracy in production

3. FAILURE MODES IDENTIFIED:
   - Ambiguous queries (e.g., "gimana" - could be order or product)
   - Heavy abbreviations without context
   - Indirect questions that require inference

RECOMMENDATIONS:
================

1. For THESIS:
   - Report V3 OOD accuracy as primary metric
   - Include manual test results for realism
   - Discuss data quality importance

2. For DEPLOYMENT:
   - Use confidence threshold ~0.75
   - Route low-confidence queries to human
   - Collect real data for continuous improvement

3. For IMPROVEMENT:
   - Add more hard negatives for confusing patterns
   - Collect real customer queries when available
   - Consider multi-intent classification
""")

print("\nFiles generated:")
for f in OUTPUT_DIR.iterdir():
    print(f"  - {f.name}")